# EasyOCR preprocessing A/B benchmark

Runs the **same trials** under several image preprocess variants and compares speed, errors, and jersey **5** accuracy in one table.

Variants: `none`, `invert`, `clahe`, `clahe_invert`, `hsv_v`.

Run all cells top to bottom.

In [ ]:
import random
import re
import time
from pathlib import Path
from statistics import mean

from PIL import Image
import cv2
import easyocr
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
BASE_DIR = ROOT / "images" / "base"
NUMBER_DIR = ROOT / "images" / "with_number"
EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

N_SLOTS = 10
N_TRIALS = 50
P_HAS_NUMBER = 0.3
RANDOM_SEED = 42
WARMUP = True
EXPECTED_JERSEY = "5"

IMAGE_SCALE = 4
OCR_KWARGS = {
    "allowlist": "0123456789",
    "mag_ratio": 1.5,
    "contrast_ths": 0.05,
    "adjust_contrast": 0.7,
}
MIN_CONFIDENCE = 0.2
USE_GPU = None

# Preprocess variants to compare (edit list to add/remove)
VARIANTS = ["none", "invert", "clahe", "clahe_invert", "hsv_v"]


def list_images(folder: Path) -> list[Path]:
    return sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXT],
        key=lambda p: p.name.lower(),
    )


def load_rgb_scaled(path: Path) -> np.ndarray:
    im = __import__("PIL").Image.open(path).convert("RGB")
    if IMAGE_SCALE != 1:
        w, h = im.size
        im = im.resize((w * IMAGE_SCALE, h * IMAGE_SCALE), im.Resampling.LANCZOS)
    return np.array(im)


def preprocess_rgb(rgb: np.ndarray, mode: str) -> np.ndarray:
    """Return 3-channel RGB uint8 for EasyOCR."""
    if mode == "none":
        return rgb
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    if mode == "invert":
        gray = 255 - gray
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    if mode == "clahe":
        lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)
    if mode == "clahe_invert":
        out = preprocess_rgb(rgb, "clahe")
        g = cv2.cvtColor(out, cv2.COLOR_RGB2GRAY)
        return cv2.cvtColor(255 - g, cv2.COLOR_GRAY2RGB)
    if mode == "hsv_v":
        v = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)[:, :, 2]
        return cv2.cvtColor(v, cv2.COLOR_GRAY2RGB)
    raise ValueError(f"Unknown preprocess mode: {mode}")


def parse_jersey_label(ocr_text: str) -> str:
    digits = re.sub(r"\D", "", ocr_text)
    return digits if digits else "(none)"


def parse_raw_detections(raw) -> list[dict]:
    out = []
    for item in raw:
        conf = float(item[2])
        text = str(item[1]).strip()
        if conf < MIN_CONFIDENCE or not text:
            continue
        out.append({"bbox": item[0], "text": text, "confidence": conf})
    return out


def read_frame(reader, rgb: np.ndarray) -> tuple[str, float, str]:
    t0 = time.perf_counter()
    raw = reader.readtext(rgb, detail=1, paragraph=False, **OCR_KWARGS)
    ms = (time.perf_counter() - t0) * 1000.0
    detections = parse_raw_detections(raw)
    text = " | ".join(d["text"] for d in detections) if detections else ""
    return text, ms, parse_jersey_label(text)


def label_error(expected: str, predicted: str, should_have_number: bool) -> str | None:
    if should_have_number:
        if predicted == "(none)":
            return "none"
        if predicted != expected:
            return "wrong"
        return None
    if predicted != "(none)":
        return "false_positive"
    return None


def build_trial(rng, bases: list[Path], numbers: list[Path]):
    paths = list(bases)
    injected = rng.random() < P_HAS_NUMBER
    slot = None
    if injected:
        slot = rng.randrange(N_SLOTS)
        paths[slot] = rng.choice(numbers)
    return paths, injected, slot


In [ ]:
bases = list_images(BASE_DIR)
numbers = list_images(NUMBER_DIR)
if len(bases) != N_SLOTS:
    raise SystemExit(f"Need {N_SLOTS} base images, found {len(bases)}")
if not numbers:
    raise SystemExit(f"Need at least 1 image in {NUMBER_DIR}")

gpu = USE_GPU
if gpu is None:
    try:
        import torch
        gpu = torch.cuda.is_available()
    except ImportError:
        gpu = False

print("Loading EasyOCR (once for all variants)...")
t0 = time.perf_counter()
reader = easyocr.Reader(["en"], gpu=gpu, verbose=False)
init_s = time.perf_counter() - t0
print(f"Reader ready in {init_s:.1f}s (gpu={gpu})")

# Same trial schedule for every variant
rng = random.Random(RANDOM_SEED)
trials = [build_trial(rng, bases, numbers) for _ in range(N_TRIALS)]

# Warmup on first base frame
if WARMUP:
    read_frame(reader, preprocess_rgb(load_rgb_scaled(bases[0]), "none"))

results: list[dict] = []

for variant in VARIANTS:
    print(f"\n--- Running variant: {variant} ---")
    frame_ms: list[float] = []
    batch_ms: list[float] = []
    tp = fp = fn = tn = 0
    preflight_ok = 0
    preflight_miss = 0
    preflight_wrong = 0

    for path in numbers:
        rgb = preprocess_rgb(load_rgb_scaled(path), variant)
        _, _, predicted = read_frame(reader, rgb)
        err = label_error(EXPECTED_JERSEY, predicted, should_have_number=True)
        if err is None:
            preflight_ok += 1
        elif err == "none":
            preflight_miss += 1
        else:
            preflight_wrong += 1

    for paths, injected, inject_slot in trials:
        t_batch = time.perf_counter()
        for slot, path in enumerate(paths):
            rgb = preprocess_rgb(load_rgb_scaled(path), variant)
            _, ms, predicted = read_frame(reader, rgb)
            frame_ms.append(ms)
            is_number_slot = injected and slot == inject_slot
            if is_number_slot and predicted == EXPECTED_JERSEY:
                tp += 1
            elif is_number_slot:
                fn += 1
            elif predicted != "(none)":
                fp += 1
            else:
                tn += 1
        batch_ms.append((time.perf_counter() - t_batch) * 1000.0)

    n_number = tp + fn
    n_plain = fp + tn
    results.append({
        "preprocess": variant,
        "mean_frame_ms": round(mean(frame_ms), 1),
        "mean_batch_ms": round(mean(batch_ms), 1),
        "batch_fps": round(1000.0 * N_SLOTS / mean(batch_ms), 2),
        "type_i_count": fp,
        "type_i_pct": round(100.0 * fp / n_plain, 1) if n_plain else 0.0,
        "type_ii_count": fn,
        "type_ii_pct": round(100.0 * fn / n_number, 1) if n_number else 0.0,
        "true_positive": tp,
        "true_negative": tn,
        "number_slot_accuracy_pct": round(100.0 * tp / n_number, 1) if n_number else 0.0,
        "preflight_ok": preflight_ok,
        "preflight_total": len(numbers),
        "preflight_ok_pct": round(100.0 * preflight_ok / len(numbers), 1),
        "preflight_miss": preflight_miss,
        "preflight_wrong": preflight_wrong,
    })
    print(f"  preflight {preflight_ok}/{len(numbers)} ok | mean frame {mean(frame_ms):.1f} ms")

print("\nAll variants finished.")


In [ ]:
ab = pd.DataFrame(results).set_index("preprocess")

# Rank helpers (higher accuracy / lower error / lower latency = better)
ab_display = ab.copy()
ab_display["rank_speed"] = ab["mean_frame_ms"].rank(method="min")
ab_display["rank_accuracy"] = ab["number_slot_accuracy_pct"].rank(ascending=False, method="min")
ab_display["rank_preflight"] = ab["preflight_ok_pct"].rank(ascending=False, method="min")
ab_display["rank_type_i"] = ab["type_i_pct"].rank(method="min")
ab_display["rank_type_ii"] = ab["type_ii_pct"].rank(method="min")

print("=" * 72)
print("PREPROCESSING A/B — side-by-side (same trials, seed=%d, n=%d)" % (RANDOM_SEED, N_TRIALS))
print("=" * 72)
display(ab.style.format(precision=1).highlight_max(
    subset=["number_slot_accuracy_pct", "preflight_ok_pct", "batch_fps"],
    color="#d4edda",
).highlight_min(
    subset=["mean_frame_ms", "type_i_pct", "type_ii_pct", "preflight_miss", "preflight_wrong"],
    color="#d4edda",
))

print("\nRank columns (1 = best among variants):")
display(ab_display[[
    "rank_speed", "rank_preflight", "rank_accuracy", "rank_type_i", "rank_type_ii"
]].astype(int))

# Best per metric
best = {
    "fastest_frame_ms": ab["mean_frame_ms"].idxmin(),
    "best_preflight": ab["preflight_ok_pct"].idxmax(),
    "best_number_slots": ab["number_slot_accuracy_pct"].idxmax(),
    "lowest_type_i": ab["type_i_pct"].idxmin(),
    "lowest_type_ii": ab["type_ii_pct"].idxmin(),
}
print("\nBest variant per metric:")
for k, v in best.items():
    print(f"  {k}: {v}")

# Delta vs baseline `none`
if "none" in ab.index:
    base = ab.loc["none"]
    delta = ab.sub(base)
    delta.columns = [f"delta_{c}" for c in delta.columns]
    print("\nDelta vs preprocess=none (negative ms / error = improvement):")
    display(delta.drop(index="none", errors="ignore").style.format(precision=1))


In [ ]:
# Preflight per image × variant (which files work under which preprocess)
rows = []
for path in numbers:
    row = {"image": path.name}
    for variant in VARIANTS:
        rgb = preprocess_rgb(load_rgb_scaled(path), variant)
        _, _, pred = read_frame(reader, rgb)
        row[variant] = pred
    rows.append(row)

preflight_grid = pd.DataFrame(rows).set_index("image")
print("Preflight predicted label per image (expected %s):" % EXPECTED_JERSEY)
display(preflight_grid)

if len(preflight_grid.columns):
    miss_count = (preflight_grid != EXPECTED_JERSEY).sum()
    print("\nMiss/wrong count per variant:")
    display(miss_count.to_frame("not_correct").sort_values("not_correct"))
